<a href="https://colab.research.google.com/github/carrascalalvarezmaria-alt/Business-Analytics-Mar-a-Pablo/blob/main/Practica_centros_alumnado_v02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Python en práctica: una red de centros deportivos
**MABA · Sesión 2 · Versión 02**

Trabajarás con **seis centros en Madrid, Barcelona y Valencia**, cuatro tipos de actividad y turnos de mañana y tarde. Cada centro ofrece tres tipos de actividad. Leerás archivos, consultarás Open-Meteo y unirás las fuentes para calcular y representar resultados.

El catálogo y la actividad son **datos sintéticos docentes**, preparados y completos. La meteorología es histórica y procede de Open-Meteo. Las coordenadas son referencias de cada ciudad, compartidas por sus dos centros; no corresponden a instalaciones reales. Las visitas cuentan asistencias a sesiones, no personas únicas.

## Cómo trabajar con las celdas
- Una celda de código **sin comentarios** está resuelta: ejecútala y observa su resultado.
- Si debes completar o cambiar código, encontrarás **`# MODIFICAR: ...`**, con la instrucción en mayúsculas. Cambia únicamente lo indicado. Los nombres exactos de variables y funciones aparecen en el enunciado; respeta sus minúsculas en Python.
- En las plantillas, sustituye los `None` o `pass`. Cuando esté resuelta la tarea, elimina también la línea `raise NotImplementedError(...)`, tal como indica su comentario. Esa línea señala una tarea pendiente.
- Las explicaciones y las preguntas están en las celdas de texto. Escribe tus respuestas donde aparece **Respuesta: …**.
- Ejecuta las celdas en orden. El notebook inicial tiene tareas pendientes; el completado debe poder ejecutarse entero.

La primera y la última respuesta son individuales y sin IA. Durante el desarrollo puedes consultar IA sobre fragmentos concretos; comprueba y explica las sugerencias que aceptes.

## Qué entregar
Notebook completado con salidas y respuestas, ZIP de resultados, enlace al repositorio y al commit final. El repositorio debe conservar una versión inicial y otra final del mismo notebook, las entradas y un README. Documenta dos decisiones de revisión con IA.

**Autores:** …  **Repositorio:** …

## 0. Entradas y diagnóstico
Descomprime el paquete. Abre el notebook en Google Colab, guarda una copia en Drive y conecta el entorno. Cuando se solicite, selecciona `actividad.csv` y `centros.xlsx` de la carpeta `datos`.

| Archivo | Campos | Qué representan |
|---|---|---|
| actividad.csv | actividad_id | Identificador único de una sesión deportiva. |
| actividad.csv | centro_id, fecha | Centro y día de la sesión. La fecha es un texto año-mes-día. |
| actividad.csv | actividad, turno | Tenis, Atletismo, Yoga o Entrenamiento funcional; Mañana o Tarde. |
| actividad.csv | duracion_min | Duración de la sesión, en minutos. |
| actividad.csv | plazas, visitas | Capacidad de la sesión y asistencias registradas. |
| centros.xlsx, hoja centros | centro_id, nombre, ciudad | Una fila por centro. |
| centros.xlsx | latitud, longitud | Coordenadas de referencia de su ciudad. |
| Open-Meteo | time, temperature_2m_max/min, precipitation_sum | Día, temperaturas máxima/mínima (°C) y precipitación (mm). |

Hay 504 sesiones: 14 días × 6 centros × 3 actividades por centro × 2 turnos. Las fechas vienen ya preparadas en el mismo formato que la API y se usarán directamente en la unión.

### Diagnóstico individual, antes de usar IA
1. ¿Qué representa una fila de actividad y una del catálogo? ¿Qué campo las relaciona?
2. Si dos centros pertenecen a una ciudad, ¿cuántas consultas meteorológicas necesitas para esa ciudad?
3. Una sesión tiene 15 visitas y 20 plazas. ¿Cómo calcularías su porcentaje de ocupación?

**Respuesta:** …

### Preparar bibliotecas
Importamos las herramientas de lectura, peticiones, cálculo y gráficos. `Path` permite construir rutas de archivos.

In [1]:
from pathlib import Path
import json
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

### Cargar las entradas en Colab
Selecciona los dos archivos de datos a la vez. Repite la carga si inicias un entorno nuevo.

In [3]:
from google.colab import files

archivos_subidos = files.upload()

Saving actividad.csv to actividad.csv
Saving centros.xlsx to centros (1).xlsx


In [4]:
DATOS = Path('.')
SALIDAS = Path('salidas')
SALIDAS.mkdir(exist_ok=True)

### Versión inicial en GitHub
Crea un repositorio `practica-centros-python` con README. Desde Colab, usa **Archivo > Guardar una copia en GitHub**. Selecciona la rama y la ruta `practica_centros.ipynb`, con mensaje `Añade punto de partida de la práctica`. Conserva ese nombre y esa ruta para guardar la versión final.

Puedes usar un repositorio privado. No necesitas GitHub Desktop ni hacer push desde el ordenador.

## 1. Funciones y bucles
### R1. Revisar una función
Este fragmento muestra el porcentaje, pero no lo devuelve para reutilizarlo:
```python
def calcular_ocupacion(visitas, plazas):
    print(visitas / plazas * 100)
```
Completa la función con `return`. Prueba `calcular_ocupacion(15, 20)`: debe devolver 75.0. Las plazas de este ejercicio son siempre positivas.

In [15]:
def calcular_ocupacion(visitas, plazas):
    return visitas / plazas * 100
print(calcular_ocupacion(15, 20))






75.0


### T1. Clasificar la ocupación
Define `clasificar_ocupacion(porcentaje)` con `if`, `elif`, `else` y `return`. Devuelve `Baja` si el porcentaje es menor que 50, `Media` si es menor que 80 y `Alta` en el resto de casos. Son categorías descriptivas del ejercicio. Comprueba los valores 40, 50 y 80.

In [16]:
def clasificar_ocupacion(porcentaje): #estamos creando una funcion
    if porcentaje < 50:
        return 'Baja'
    elif porcentaje < 80:
        return 'Media'
    else:
        return 'Alta'
    pass
print(clasificar_ocupacion(40))
print(clasificar_ocupacion(50))
print(clasificar_ocupacion(80))

Baja
Media
Alta


### R2. Revisar un while
Lee este fragmento sin ejecutarlo: le falta avanzar el índice.
```python
turnos = ['Mañana', 'Tarde']
indice = 0
while indice < len(turnos):
    print(turnos[indice])
```
Completa el bucle para que muestre cada turno una sola vez y termine. **Respuesta:** ¿qué ocurre si el índice nunca cambia?
#Nos devolvería mañana infinitas veces porque siempre se cumpliria la condicion  while indice < len(turnos):

In [17]:
turnos = ['Mañana', 'Tarde']
indice = 0
# MODIFICAR: COMPLETA EL BUCLE WHILE Y AUMENTA EL ÍNDICE EN CADA VUELTA.
while indice < len(turnos):
    print(turnos[indice])
    indice += 1



Mañana
Tarde


## 2. Leer archivos y utilizar estructuras
### T2. CSV y Excel
Lee `actividad.csv` con `pd.read_csv` y `centros.xlsx` con `pd.read_excel`, indicando `sheet_name='centros'`. Usa las rutas `DATOS / 'actividad.csv'` y `DATOS / 'centros.xlsx'`.

In [26]:
# MODIFICAR: LEE EL CSV Y GUARDA SU TABLA EN LA VARIABLE INDICADA.
actividad_original= pd.read_csv(DATOS / 'actividad.csv')
# MODIFICAR: LEE EL EXCEL Y GUARDA SU TABLA EN LA VARIABLE INDICADA.
centros = pd.read_excel(DATOS / 'centros.xlsx', sheet_name='centros')
# MODIFICAR: LEE EL EXCEL E INDICA LA HOJA DEL CATÁLOGO.


In [27]:
print(actividad_original.shape)
print(actividad_original.columns)
print(actividad_original.dtypes)
actividad_original.head()


(504, 8)
Index(['actividad_id', 'centro_id', 'fecha', 'actividad', 'turno',
       'duracion_min', 'plazas', 'visitas'],
      dtype='object')
actividad_id    object
centro_id       object
fecha           object
actividad       object
turno           object
duracion_min     int64
plazas           int64
visitas          int64
dtype: object


,actividad_id,centro_id,fecha,actividad,turno,duracion_min,plazas,visitas
0,ACT0001,C01,2026-09-01,Atletismo,Mañana,75,32,22
1,ACT0002,C01,2026-09-01,Atletismo,Tarde,75,32,20
2,ACT0003,C01,2026-09-01,Yoga,Mañana,45,24,14
3,ACT0004,C01,2026-09-01,Yoga,Tarde,45,24,19
4,ACT0005,C01,2026-09-01,Entrenamiento funcional,Mañana,60,20,10


In [ ]:
actividad_original.info()
centros

### Etiquetas y posiciones
Ejecuta los ejemplos. `loc` usa etiquetas y nombres de columnas; `iloc` usa posiciones. **Respuesta:** ¿qué devuelve cada celda? …
#LOC devuelve el primer valor de actividad id y de centro id (C01)(ACT0001)
#ILOC:nos devuelve el primer valor de la segunda línea (CO1)


In [28]:
actividad_original.loc[0, ['actividad_id', 'centro_id']]#devuelve el primer valor de actividad id y de centro id (C01)(ACT0001)

,0
actividad_id,ACT0001
centro_id,C01


In [29]:
actividad_original.iloc[0, 1] #nos devuelve el primer valor de la segunda línea (CO1)

'C01'

### T3. Listas, diccionarios, tuplas y conjuntos
`to_dict('records')` produce una lista de diccionarios, uno por centro. Completa una comprensión para obtener sus ciudades. Después utiliza `set` para obtener las ciudades distintas. Guarda las coordenadas del primer centro en una tupla y desempaquétala en `latitud` y `longitud`.

**Respuesta:** ¿por qué la lista tiene seis elementos y el conjunto solo tres?
#Porque ciudades_lista guarda todos los elementos como aparecen, permitiendo duplicados. Si tienes 6 centros en total, la lista tendrá 6 ciudades. Mientras que el conjunto set elimina duplicados y solo almcena valores únicos entonces a pesar de tener 6 centros solo diponemos de 3 ciudades diferentes que son Madrid, Barcelona y Valencia.


In [30]:
centros_lista = centros.to_dict('records')
centros_lista

[{'centro_id': 'C01',
  'nombre': 'Centro Retiro',
  'ciudad': 'Madrid',
  'latitud': 40.4168,
  'longitud': -3.7038},
 {'centro_id': 'C02',
  'nombre': 'Centro Alameda',
  'ciudad': 'Madrid',
  'latitud': 40.4168,
  'longitud': -3.7038},
 {'centro_id': 'C03',
  'nombre': 'Centro Litoral',
  'ciudad': 'Barcelona',
  'latitud': 41.3851,
  'longitud': 2.1734},
 {'centro_id': 'C04',
  'nombre': 'Centro Montjuïc',
  'ciudad': 'Barcelona',
  'latitud': 41.3851,
  'longitud': 2.1734},
 {'centro_id': 'C05',
  'nombre': 'Centro Turia',
  'ciudad': 'Valencia',
  'latitud': 39.4699,
  'longitud': -0.3763},
 {'centro_id': 'C06',
  'nombre': 'Centro Malvarrosa',
  'ciudad': 'Valencia',
  'latitud': 39.4699,
  'longitud': -0.3763}]

In [34]:
# MODIFICAR: OBTÉN LA LISTA DE CIUDADES MEDIANTE UNA COMPRENSIÓN.
ciudades = set(centro['ciudad'] for centro in centros_lista)
print (ciudades)
# MODIFICAR: CREA UN CONJUNTO A PARTIR DE LA LISTA DE CIUDADES.
ciudades_distintas = set(ciudades)
print(ciudades_distintas)
# MODIFICAR: CREA UNA TUPLA CON LATITUD Y LONGITUD DEL PRIMER CENTRO Y DESEMPAQUÉTALA.
coordenadas = centros_lista[0]['latitud'], centros_lista[0]['longitud']
latitud, longitud = coordenadas
print(coordenadas)

{'Barcelona', 'Madrid', 'Valencia'}
{'Barcelona', 'Madrid', 'Valencia'}
(40.4168, -3.7038)


## 3. Consultar la API
### T4. Petición y JSON
La función consulta los 14 días de una ciudad. Completa la petición con `requests.get(url, params=parametros, timeout=30)` y convierte `contenido['daily']` en una tabla con `pd.DataFrame`. Lee los parámetros y las unidades de la respuesta.

In [38]:
def consultar_meteo(ciudad, latitud, longitud):
    url = 'https://archive-api.open-meteo.com/v1/archive'
    variables = 'temperature_2m_max,temperature_2m_min,precipitation_sum'
    parametros = {
        'latitude': latitud,
        'longitude': longitud,
        'start_date': '2026-09-01',
        'end_date': '2026-09-14',
        'daily': variables,
        'timezone': 'Europe/Madrid'
    }
    # MODIFICAR: REALIZA LA PETICIÓN CON LA URL, LOS PARÁMETROS Y EL TIEMPO DE ESPERA INDICADOS.
    respuesta = requests.get(url, params=parametros, timeout=30)
    # MODIFICAR: CONVIERTE LA RESPUESTA EN UNA TABLA.
    print(ciudad, respuesta.status_code)
    respuesta.raise_for_status()
    contenido = respuesta.json()
    print(contenido['daily_units'])
    tabla_ciudad = pd.DataFrame(contenido['daily'])
    tabla_ciudad = tabla_ciudad.rename(columns={'time': 'fecha'})
    return tabla_ciudad
    tabla_ciudad['ciudad'] = ciudad


### Una consulta por ciudad
El diccionario `ciudades_api` guarda unas coordenadas por ciudad. Los dos centros de cada ciudad comparten la referencia meteorológica. Asignar de nuevo la misma clave de un diccionario no crea otra entrada.

El `for` consulta cada ciudad y `append` añade su tabla a una lista. `concat` apila las tres tablas. El resultado tiene 42 filas: 3 ciudades × 14 días. Si la API no responde, el docente dispone de un respaldo para continuar.

In [ ]:
ciudades_api = {}
for centro in centros_lista:
    ciudades_api[centro['ciudad']] = (centro['latitud'], centro['longitud'])

ciudades_api

In [ ]:
tablas_meteo = []
for ciudad, coordenadas in ciudades_api.items():
    latitud, longitud = coordenadas
    tabla_ciudad = consultar_meteo(ciudad, latitud, longitud)
    tablas_meteo.append(tabla_ciudad)

meteo = pd.concat(tablas_meteo, ignore_index=True)
print(meteo.shape)
meteo.head()

**Respuesta:** ¿qué contiene `daily`? ¿Dónde aparecen las unidades? ¿Por qué la meteorología tiene menos filas que la actividad? …

## 4. Unir las fuentes y calcular nuevas columnas
### T5. Dos uniones
Completa dos `merge` con `how='left'`. Primero une `actividad_original` con `centros` mediante `centro_id`. Después une el resultado con `meteo` mediante `['ciudad', 'fecha']`.

La primera unión añade nombre, ciudad y coordenadas. La segunda añade los datos meteorológicos correspondientes al día y ciudad de cada sesión. Las claves de las tablas auxiliares identifican una sola fila. Las sesiones de un mismo día y ciudad comparten meteorología.

**Respuesta:** ¿por qué la segunda unión necesita tanto ciudad como fecha? …

In [ ]:
# MODIFICAR: UNE LA ACTIVIDAD CON EL CATÁLOGO POR EL CÓDIGO DEL CENTRO.
preparada = None
# MODIFICAR: UNE EL RESULTADO CON LA METEOROLOGÍA POR CIUDAD Y FECHA.
preparada = None
# MODIFICAR: ELIMINA LA SIGUIENTE LÍNEA TRAS COMPLETAR LAS DOS UNIONES.
raise NotImplementedError('T5')

In [ ]:
print('Sesiones antes de unir:', len(actividad_original))
print('Sesiones después de unir:', len(preparada))
preparada.head()

### T6. Ocupación y duración en horas
Utiliza `calcular_ocupacion` con las columnas `preparada['visitas']` y `preparada['plazas']`. La misma operación aritmética se aplica a cada fila. Guarda el resultado en `ocupacion_pct`. Divide `duracion_min` entre 60 y guarda `duracion_horas`.

La celda siguiente utiliza `apply` con la función de clasificación que has definido.

In [ ]:
# MODIFICAR: CALCULA EL PORCENTAJE DE OCUPACIÓN CON LA FUNCIÓN DEFINIDA.
preparada['ocupacion_pct'] = None
# MODIFICAR: CONVIERTE LA DURACIÓN DE MINUTOS A HORAS.
preparada['duracion_horas'] = None
# MODIFICAR: ELIMINA LA SIGUIENTE LÍNEA TRAS COMPLETAR LOS DOS CÁLCULOS.
raise NotImplementedError('T6')

In [ ]:
preparada['nivel_ocupacion'] = preparada['ocupacion_pct'].apply(clasificar_ocupacion)
preparada[['actividad', 'turno', 'visitas', 'plazas', 'ocupacion_pct', 'nivel_ocupacion']].head()

## 5. Resumir y visualizar
### NumPy y SciPy
La amplitud térmica es la máxima menos la mínima del día. La calculamos con arrays y obtenemos su media sobre la tabla de meteorología, donde cada ciudad-día aparece una vez.

`stats.iqr` devuelve la diferencia entre los percentiles 75 y 25. Aquí mide la dispersión de las visitas por sesión. Ejecuta y explica qué aportan la media, los cuartiles y el rango intercuartílico.

In [ ]:
maximas = np.array(meteo['temperature_2m_max'])
minimas = np.array(meteo['temperature_2m_min'])
meteo['amplitud_termica'] = maximas - minimas
print('Amplitud térmica media (°C):', np.mean(meteo['amplitud_termica']))

In [ ]:
print('Rango intercuartílico de visitas:', stats.iqr(preparada['visitas']))
preparada['visitas'].describe()

### T7. Agrupar por centro y por actividad y turno
Completa `resumen`, la suma de visitas por `nombre`, con `groupby` y `sum`. En `ocupacion_por_turno`, calcula la media de `ocupacion_pct` agrupando por `['actividad', 'turno']`.

La segunda tabla da el porcentaje medio por sesión: cada sesión pesa lo mismo. Si quisieras el porcentaje de todas las plazas ocupadas, el cálculo sería 100 × suma de visitas / suma de plazas.

**Respuesta:** ¿qué pregunta responde cada tabla? …

In [ ]:
# MODIFICAR: SUMA LAS VISITAS AGRUPANDO POR EL NOMBRE DEL CENTRO.
resumen = None
# MODIFICAR: CALCULA LA OCUPACIÓN MEDIA POR TIPO DE ACTIVIDAD Y TURNO.
ocupacion_por_turno = None
# MODIFICAR: ELIMINA LA SIGUIENTE LÍNEA TRAS COMPLETAR LAS DOS AGRUPACIONES.
raise NotImplementedError('T7')

### T8. Tabla dinámica
Construye `visitas_diarias` con `pivot_table`: `fecha` en filas, `ciudad` en columnas y suma de `visitas` como valores. La usaremos para representar tres series, una por ciudad.

In [ ]:
# MODIFICAR: CREA LA TABLA DINÁMICA DE VISITAS POR FECHA Y CIUDAD.
visitas_diarias = None
# MODIFICAR: ELIMINA LA SIGUIENTE LÍNEA TRAS COMPLETAR LA TABLA.
raise NotImplementedError('T8')

### T9. Gráficos con títulos y unidades
Completa los títulos y los nombres de los ejes. La primera figura representa el total diario de visitas por ciudad. La segunda muestra la distribución del porcentaje de ocupación por tipo de actividad y turno.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
visitas_diarias.plot(ax=ax, marker='o')
# MODIFICAR: ESCRIBE UN TÍTULO QUE DESCRIBA EL GRÁFICO.
ax.set_title('')
# MODIFICAR: IDENTIFICA LA VARIABLE DEL EJE HORIZONTAL.
ax.set_xlabel('')
# MODIFICAR: IDENTIFICA LA VARIABLE Y UNIDAD DEL EJE VERTICAL.
ax.set_ylabel('')
ax.set_ylim(bottom=0)
ax.legend(title='Ciudad')
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
fig.savefig(SALIDAS / 'visitas_diarias.png', dpi=150)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.boxplot(data=preparada, x='actividad', y='ocupacion_pct', hue='turno', ax=ax)
# MODIFICAR: ESCRIBE UN TÍTULO QUE DESCRIBA EL GRÁFICO.
ax.set_title('')
# MODIFICAR: IDENTIFICA LA VARIABLE DEL EJE HORIZONTAL.
ax.set_xlabel('')
# MODIFICAR: IDENTIFICA LA VARIABLE Y UNIDAD DEL EJE VERTICAL.
ax.set_ylabel('')
ax.set_ylim(0, 100)
ax.legend(title='Turno')
fig.tight_layout()
fig.savefig(SALIDAS / 'ocupacion_actividad.png', dpi=150)
plt.show()

**Respuesta:** escribe una observación descriptiva de cada gráfico. Distingue visitas totales y porcentaje de ocupación. Explica por qué estos datos sintéticos no permiten atribuir a la meteorología cambios en la asistencia. …

## 6. Revisar código con IA y guardar resultados
### Revisar una propuesta
Un asistente propone unir la meteorología solo mediante `ciudad`:
```python
preparada = actividad_original.merge(centros, on='centro_id', how='left')
preparada = preparada.merge(meteo, on='ciudad', how='left')
```
Lee la propuesta y compárala con T5. ¿Qué ocurre cuando una ciudad tiene 14 días meteorológicos? Explica por qué añadir `fecha` a la clave cambia el resultado. Puedes comprobarlo en una tabla temporal sin sustituir `preparada`.

Después, solicita a la IA que revise una función o agrupación concreta. Dale el objetivo, las columnas y el fragmento. Comprueba su sugerencia antes de incorporarla.

| Decisión | Consulta y sugerencia | Qué aceptaste o cambiaste | Comprobación y resultado |
|---|---|---|---|
| 1 | … | … | … |
| 2 | … | … | … |

No hace falta entregar conversaciones completas.

### Exportar resultados
Se guarda la actividad enriquecida, la meteorología por ciudad-día, los resúmenes y los dos gráficos. Las entradas permanecen separadas.

In [ ]:
preparada.to_csv(SALIDAS / 'actividad_preparada.csv', index=False)
meteo.to_csv(SALIDAS / 'meteo_diaria.csv', index=False)
resumen.to_csv(SALIDAS / 'resumen_centros.csv')
ocupacion_por_turno.to_csv(SALIDAS / 'ocupacion_por_turno.csv')
visitas_diarias.to_csv(SALIDAS / 'visitas_diarias.csv')

In [ ]:
ficha = {
    'actividad': 'Datos sintéticos docentes MABA',
    'meteorologia': 'API histórica de Open-Meteo',
    'periodo': ['2026-09-01', '2026-09-14'],
    'ciudades': list(ciudades_api),
    'variables': 'temperature_2m_max,temperature_2m_min,precipitation_sum'
}
with open(SALIDAS / 'fuentes.json', 'w', encoding='utf-8') as archivo:
    json.dump(ficha, archivo, ensure_ascii=False, indent=2)

### Comprobación de resultados
Las uniones deben conservar las 504 sesiones. La suma de visitas debe coincidir en la entrada, la tabla preparada y los resúmenes. Revisa los títulos y las unidades de los gráficos y la fuente realmente utilizada.

In [ ]:
print('Sesiones de entrada:', len(actividad_original))
print('Sesiones tras unir:', len(preparada))
print('Visitas de entrada:', actividad_original['visitas'].sum())
print('Visitas tras unir:', preparada['visitas'].sum())
print('Visitas del resumen:', resumen.sum())
print('Visitas de la tabla diaria:', visitas_diarias.sum().sum())

In [ ]:
import shutil

shutil.make_archive('resultados_centros', 'zip', SALIDAS)
files.download('resultados_centros.zip')

### README y versión final en GitHub
Completa la plantilla `README_plantilla.md` en el README de tu repositorio. Describe las fuentes, las entradas, los pasos para cargar archivos en Colab y las salidas. Registra las dos decisiones sobre IA.

Guarda el notebook desde Colab en **el mismo repositorio, rama y ruta `practica_centros.ipynb`** de la versión inicial. Usa un mensaje como `Completa funciones, uniones y resultados`. En GitHub, localiza el historial y explica una diferencia entre las dos versiones.

Guardar en Drive no actualiza GitHub. Guardar en GitHub tampoco incorpora los CSV de ejecución. Descarga el ZIP y adjúntalo a la entrega. Para reproducir el proyecto desde el repositorio, sube también el CSV y el Excel de entrada desde la web a una carpeta `datos`.

**Entrega:** enlace al repositorio, enlace al commit final y ZIP de resultados.

## 7. Comprobación individual final, sin IA
1. Explica una modificación de código, su propósito y cómo la comprobaste.
2. ¿Por qué tenemos 504 sesiones, 6 centros y 42 filas meteorológicas? ¿Qué representa una fila en cada caso?
3. Explica las claves de las dos uniones. ¿Por qué no basta con unir meteorología por ciudad?
4. ¿Qué devuelven `calcular_ocupacion(18, 24)` y `clasificar_ocupacion(80)`? Explica cómo se obtienen.
5. ¿Qué cambiarías en T7 para obtener visitas totales por turno?

**Nombre y respuesta individual:** …

## Referencias
- [Open-Meteo: API histórica y variables](https://open-meteo.com/en/docs/historical-weather-api).
- [Requests: peticiones y respuestas](https://requests.readthedocs.io/en/latest/user/quickstart/).
- [pandas: merge](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.merge.html).
- [SciPy: rango intercuartílico](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.iqr.html).
- [Colab: archivos y entorno temporal](https://research.google.com/colaboratory/faq.html).

Los datos meteorológicos pueden revisarse en el servicio. Se comprueban estructura, fechas y unidades, sin exigir valores meteorológicos inmutables.